# SARBLOH — Alignment: LoRA SFT + LoRA RL (GRPO)

One notebook, two pipelines. Each writes a result card with a PASS/FAIL verdict built from explicit checks:

1. **SFT** (`Alignment/lora_sft.py`, E107). Loss is masked to the assistant tokens. The LoRA adapter is saved,
   reloaded into a fresh base, and must reproduce the eval loss. Output: `/kaggle/working/sft_out/`.
2. **RL** (`Alignment/lora_rl.py`, E108). GRPO uses a verifiable tool-call reward (`tool_match`) or your own
   `module:function`, with KL to the adapter-disabled reference. It can start from the SFT adapter
   (`init_adapter`). Output: `/kaggle/working/rl_out/`.

**Workflow.** This file is the Kaggle notebook. Edit it directly, then run
`uv run python scripts/kaggle_push.py alignment`. That command refreshes the generated source cells below from
`Alignment/*.py` and `Alignment/data/`, then pushes this exact file. Afterwards, `alignment-status` and
`alignment-output` fetch the results.

**To train on your own data**, set `DATA` to a `/kaggle/input/...` JSONL file and attach that dataset under
`alignment.datasets` in `scripts/kaggle_settings.json`. Prime `sft_levels.jsonl` rows work as they are. Also set
`MODEL`, raise the steps and lower the learning rates. Switch off either pipeline with `RUN_SFT` / `RUN_RL`.

In [ ]:
# Environment, and dependencies checked in a subprocess so an upgrade is picked up before transformers is imported.
import json, subprocess, sys, time
T0 = time.time()
subprocess.run(["nvidia-smi"], check=False)
PROBE = """
import json
out = {}
try:
    import transformers
    from transformers import CONFIG_MAPPING
    out["transformers"] = transformers.__version__
    out["qwen3_5"] = "qwen3_5" in CONFIG_MAPPING
except Exception:
    out["transformers"], out["qwen3_5"] = None, False
try:
    import peft
    out["peft"] = peft.__version__
except Exception:
    out["peft"] = None
import torch
out["torch"], out["cuda"] = torch.__version__, torch.cuda.is_available()
print(json.dumps(out))
"""

def probe():
    return json.loads(subprocess.run([sys.executable, "-c", PROBE], capture_output=True, text=True,
                                     check=True).stdout.strip().splitlines()[-1])

deps = probe()
print("before:", deps)
need = []
if not deps["qwen3_5"]:
    need.append("transformers>=5.0")
if not deps["peft"]:
    need.append("peft")
if need:  # needs internet: this kernel runs with internet on
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *need], check=True)
    deps = probe()
    print("after: ", deps)
assert deps["qwen3_5"] and deps["peft"] and deps["cuda"], deps

In [ ]:
# @@ALIGNMENT_SOURCES@@ placeholder: `scripts/kaggle_push.py alignment` replaces this cell and the %%writefile cells after it.

In [ ]:
import gc
from pathlib import Path
from IPython.display import Markdown, display
SRC = Path("alignment_src").resolve()
sys.path.insert(0, str(SRC))
SOURCES_META = json.loads((SRC / "SOURCES_META.json").read_text(encoding="utf-8"))
print(SOURCES_META)
import lora_sft, lora_rl

def free_gpu():
    import torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

def show(card_dir):
    display(Markdown(Path(card_dir, "result_card.md").read_text(encoding="utf-8")))

In [ ]:
# Shared settings: both pipelines read the same model and data.
MODEL = "Qwen/Qwen3.5-2B"                 # same qwen3_5 family and chat template as Qwen3.8-27B; or a local weights dir
DATA = str(SRC / "data" / "smoke3.jsonl")  # chat/tool JSONL or Prime sft_levels.jsonl; real data: /kaggle/input/...
MAX_SAMPLES = 3                           # conversations taken from DATA (None = all)
OUT = Path("/kaggle/working")
RUN_SFT = True
RUN_RL = True

## 1. SFT: LoRA on the assistant turns

Keys are `lora_sft.DEFAULTS`; anything not set here takes its default.

In [ ]:
SFT_CONFIG = {
    "model": MODEL,
    "data_path": DATA,
    "max_samples": MAX_SAMPLES,
    "output_dir": str(OUT / "sft_out"),
    "dtype": "auto",                     # bf16 on sm80+, fp32 on T4 / P100
    "max_seq_len": 4096,
    "lora_r": 16, "lora_alpha": 32, "lora_dropout": 0.0,
    "lr": 5e-4, "max_steps": 40, "warmup_ratio": 0.1, "batch_size": 1, "grad_accum": 1,
    "gradient_checkpointing": True,
    "gen_samples": 3, "gen_max_new_tokens": 200,
    "min_loss_drop": 0.5,                # smoke bar: three samples must be largely memorised
    "reload_check": True,
    "seed": 0,
}
sft_card = None
if RUN_SFT:
    sft_card = lora_sft.run(SFT_CONFIG)
    free_gpu()
    show(SFT_CONFIG["output_dir"])

## 2. RL: LoRA GRPO with a verifiable reward

Keys are `lora_rl.DEFAULTS`. The reward is `tool_match`: the action is scored against the reference turn's tool
calls (stop, valid call, tool names, arguments), not the wording. Set `"reward": "module:function"` for your own.
To run RL on top of SFT, set `"init_adapter": SFT_CONFIG["output_dir"] + "/adapter"`. The SFT adapter is merged into
the base and becomes the KL reference. The smoke run starts from the base, so that the reward has room to rise.

In [ ]:
RL_CONFIG = {
    "model": MODEL,
    "init_adapter": None,                # e.g. SFT_CONFIG["output_dir"] + "/adapter" for SFT -> RL
    "data_path": DATA,
    "max_samples": MAX_SAMPLES,
    "turns": "last",                     # one prompt per conversation ("all": one per assistant turn)
    "output_dir": str(OUT / "rl_out"),
    "dtype": "auto",
    "reward": "tool_match",
    "lora_r": 16, "lora_alpha": 32, "lora_dropout": 0.0,
    "group_size": 4, "prompts_per_step": 3, "max_steps": 15,
    "lr": 2e-4, "beta": 0.02, "scale_rewards": True,
    "temperature": 1.0, "top_p": 1.0, "top_k": 0, "max_new_tokens": 160, "eval_max_new_tokens": 160,
    "gradient_checkpointing": True,
    "min_reward_gain": 0.1,              # smoke bar: greedy reward on the prompts must rise by this much
    "reload_check": True,
    "seed": 0,
}
rl_card = None
if RUN_RL:
    rl_card = lora_rl.run(RL_CONFIG)
    free_gpu()
    show(RL_CONFIG["output_dir"])

## Summary

`alignment_summary.json` holds both verdicts. The notebook fails, and Kaggle marks the run as errored, if either
pipeline that ran is FAIL.

In [ ]:
def brief(card):
    if card is None:
        return None
    out = {"verdict": card["verdict"], "checks": {k: v["pass"] for k, v in card["checks"].items()},
           "seconds": card.get("seconds_total"), "peak_vram_gb": card.get("peak_vram_gb"), "dtype": card.get("dtype"),
           "gpu": card.get("environment", {}).get("gpu")}
    if "eval_before" in card and "loss" in card["eval_before"]:
        out.update(eval_loss_before=card["eval_before"]["loss"], eval_loss_after=(card.get("eval_after") or {}).get("loss"),
                   eval_loss_reloaded=(card.get("eval_reloaded") or {}).get("loss"))
    if "eval_before" in card and "reward" in card["eval_before"]:
        out.update(reward_before=card["eval_before"]["reward"], reward_after=(card.get("eval_after") or {}).get("reward"),
                   reward_references=card.get("reference_reward"), reload=card.get("reload"))
    return out

summary = {"sft": brief(sft_card), "rl": brief(rl_card), "source": SOURCES_META, "wall_s": round(time.time() - T0, 1)}
print("ALIGNMENT_SUMMARY " + json.dumps(summary))
(OUT / "alignment_summary.json").write_text(json.dumps(summary, indent=1), encoding="utf-8")
failed = [k for k, c in (("sft", sft_card), ("rl", rl_card)) if c is not None and c["verdict"] != "PASS"]
assert not failed, f"result card verdict FAIL: {failed} (see the cards above)"